# Create American Society of Hematology (ASH) Awards

Creates the research, career-development and training awards of the **American Society of
Hematology** from ASH's own award-recipient pages.

**Prerequisite:** `scripts/local/ash_to_s3.py` (uploads the parquet).

**Source:** the recipients page of each ASH funding programme under
https://www.hematology.org/awards/ (server-rendered, listed in the site's sitemap.xml;
robots.txt only disallows Sitecore system paths). One row per recipient per year.

**2,657 awards, 1985-2026**, by programme:
- research grants (`research`): ASH Bridge Grant 209 (2013-2025 rounds), ASH Award for
  Research Careers in Hematology (ARCH, Bridge successor) 33, Global Research Award 93,
  Research Restart Award 19, Treating Fairly Award 10
- career development (`career_development`): ASH Scholar Award 733 (1985-2026, fellow /
  fellow-to-faculty / junior faculty categories from 2013), ASH-Amos Medical Faculty
  Development Award (AMFDP / AIMFD) 29
- fellowships and training (`fellowship` / `training`): Research Training Award for Fellows
  188 (the "Second Year Awardees" repeats are dropped), HONORS 427, Medical Student
  Physician-Scientist 53, Graduate Hematology 51, HIP Fellow 54, HIP Graduate Student 63,
  HIP Medical Student 326, HIP Resident 91, and the funded training visits: Visitor Training
  Program 197, Latin American Training Program 73, African Visitor Training Program 8
- The ARCH page repeats the 2025 Bridge Grant round with project titles; those 25 people
  are kept once, as Bridge Grant rows carrying the title.

**Not ingested (recognition rather than research funding; flagged for review):** honorific
prizes and lectures (Dameshek, Ranney, Rowley, Stratton, Thomas, Beutler, Ham-Wasserman,
Coulter, Mentor / Sadler-Forget, Scott-Drew / Advancing Inclusive Excellence), the Abstract
Achievement awards (formerly travel awards, incl. national-partner-society and named
abstract awards) and the CRTI / Medical Educators Institute course participant-and-faculty
lists.

**Field mapping:** `display_name` = project title where published (ARCH, Treating Fairly,
2025 Bridge round), training visits `{programme} {year}: {name} ({training topic})`, else
`{programme} {year}: {name}`; `description` = training topic + host mentor for the visit
programmes; `funder_scheme` = programme (category appended for Scholar / RTAF / HONORS);
`start_year` = list year (no dates or amounts published); `lead_investigator` = recipient
(name split in the script, degrees dropped), affiliation = institution and country where
printed.

**Amounts:** none published per award (ASH gives programme ceilings only); amount check
(6.7) waived.

**`funder_award_id` (2.1.1):** ASH publishes no award numbers (a handful of citing works
quote internal codes like `ASHAMFDP-20121` or `AGRA2023-4`), so the key is the synthetic
`ASH-{programme code}-{year}-{recipient slug}`. 0 existing citation stubs collapse.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306571`
- display_name / ror_id / doi: from `openalex.funders.funders` (ror 02nw48b86, doi 10.13039/100001422)

**Priority:** `576` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ash_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ash/ash_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total, COUNT(DISTINCT funder_award_id) as distinct_ids,
       MIN(year) as min_year, MAX(year) as max_year
FROM openalex.awards.ash_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.ash_raw;

In [ ]:
%sql
SELECT programme, funding_type, COUNT(*) AS n, MIN(year) AS first_year, MAX(year) AS last_year,
       COUNT(institution) AS with_institution, COUNT(title) AS with_title
FROM openalex.awards.ash_raw GROUP BY 1, 2 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 1, 'F4320306571 must have exactly one openalex.funders.funders row') AS funder_ok,
       MAX(display_name) AS display_name, MAX(ror_id) AS ror_id, MAX(doi) AS doi
FROM openalex.funders.funders
WHERE funder_id = 4320306571;

## Step 2: Create ASH Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ash_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306571  -- American Society of Hematology
),
g AS (
    SELECT *,
           CONCAT_WS(' ', NULLIF(TRIM(lead_given_name), ''), NULLIF(TRIM(lead_family_name), '')) AS person,
           programme_code IN ('VTP', 'LATP', 'AVTP') AS training_visit
    FROM openalex.awards.ash_raw
    WHERE funder_award_id IS NOT NULL AND TRIM(funder_award_id) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    CASE WHEN g.training_visit
         THEN CONCAT(g.programme, ' ', g.year, ': ', g.person,
                     CASE WHEN NULLIF(TRIM(g.title), '') IS NOT NULL THEN CONCAT(' (', TRIM(g.title), ')') ELSE '' END)
         ELSE COALESCE(NULLIF(TRIM(g.title), ''), CONCAT(g.programme, ' ', g.year, ': ', g.person))
    END as display_name,
    CASE WHEN g.training_visit
         THEN NULLIF(CONCAT_WS(' ', CONCAT('Training topic: ', NULLIF(TRIM(g.title), ''), '.'),
                               CONCAT('Host: ', NULLIF(TRIM(g.host), ''), '.')), '')
    END as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    g.funding_type as funding_type,
    CASE WHEN g.programme_code IN ('SCHOLAR', 'RTAF', 'HONORS') AND NULLIF(TRIM(g.category), '') IS NOT NULL
         THEN CONCAT(g.programme, ': ', TRIM(g.category))
         ELSE g.programme END as funder_scheme,
    'ash_award_recipients' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation.
    named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', NULLIF(TRIM(g.country), ''),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f
WHERE NULLIF(TRIM(g.lead_family_name), '') IS NOT NULL;

## Shape check (2.1.1)
Synthetic keys only: `ASH-{CODE}-{yyyy}-{recipient slug}`.

In [ ]:
%sql
SELECT assert_true(COUNT(*) = 0, 'unexpected funder_award_id shape') AS shape_ok
FROM openalex.awards.ash_awards
WHERE NOT funder_award_id RLIKE '^ASH-[A-Z]+-(19|20)[0-9]{2}-[a-z0-9-]+$';

In [ ]:
%sql
SELECT assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok, COUNT(*) AS n
FROM openalex.awards.ash_awards;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ash_award_recipients' AND priority = 576;

-- Priority 576: direct-from-funder ingest of ASH's own award-recipient pages.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    576 as priority
FROM openalex.awards.ash_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.ash_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, funder_scheme, COUNT(*) as cnt
FROM openalex.awards.ash_awards GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.ash_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.ash_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.ash_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- 6.3 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi
FROM openalex.awards.ash_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ash_award_recipients'
GROUP BY provenance, priority;